# 4. Microsoft GraphRAG

[GraphRAG](https://arxiv.org/abs/2404.16130) builds its index in four stages:

1. **Extract** entities and relations from each chunk, as LightRAG does.
2. **Cluster** the graph into nested communities with the Leiden algorithm.
3. **Summarise** each community into a report written by the LLM.
4. **Embed** entity descriptions, chunks and reports.

Two search modes then use this index:

- **Local search** starts from the entities closest to the question and gathers their relations, source chunks and community reports.
- **Global search** sends batches of community reports to the LLM (map), then merges the partial answers (reduce). It is designed for broad, corpus-wide questions.

In [ ]:
from dotenv import load_dotenv

from src import config

load_dotenv(config.PROJECT_ROOT / ".env")
print(f"Domain: {config.DOMAIN} | run mode: {config.RUN_MODE.value}")

In [ ]:
from src.data import load_full_corpus_statistics, load_run_inputs
from src.usage_tracking import UsageLedger, get_ledger_path, register_litellm_usage_callback

INDEX_NAME = "graphrag"

run_inputs = load_run_inputs(config.DOMAIN, config.RUN_MODE)
ledger = UsageLedger(get_ledger_path(run_inputs.run_directory))
index_directory = run_inputs.run_directory / "indexes" / INDEX_NAME
print(f"{len(run_inputs.questions)} questions, {len(run_inputs.documents)} documents")

## 4.1 Configure GraphRAG

GraphRAG reads `graphrag_project/settings.yaml` and its prompts in `graphrag_project/prompts/`. The helper below points the configuration at the documents of this run and applies the shared settings of `src/config.py`.

GraphRAG calls models through LiteLLM. Registering a LiteLLM callback sends the token usage of every call to the ledger.

In [ ]:
from src.graphrag_utils import load_graphrag_config

graphrag_config = load_graphrag_config(corpus_directory=run_inputs.corpus_directory, index_directory=index_directory)
register_litellm_usage_callback(ledger)

completion_model = graphrag_config.completion_models["default_completion_model"]
print(f"Model: {completion_model.model} {completion_model.call_args}")
print(f"Chunks: {graphrag_config.chunking.size} tokens, overlap {graphrag_config.chunking.overlap}")
print(f"Index written to: {graphrag_config.output_storage.base_dir}")

## 4.2 Build the index

`build_index` runs the whole pipeline and returns one result per workflow. GraphRAG caches every LLM response in `index_directory/cache`, so an interrupted build resumes without paying twice. Deleting `index_directory` forces a full rebuild.

In [ ]:
import time

from graphrag.api import build_index
from graphrag.config.enums import IndexingMethod

from src.usage_tracking import Phase, load_indexing_report, save_indexing_report, usage_scope

if load_indexing_report(run_inputs.run_directory, INDEX_NAME):
    print("Index found, reusing it.")
else:
    if not index_directory.exists():
        ledger.discard_records(INDEX_NAME, Phase.INDEXING)
    start_time = time.perf_counter()
    with usage_scope(INDEX_NAME, Phase.INDEXING):
        workflow_results = await build_index(config=graphrag_config, method=IndexingMethod.Standard)
    for workflow_result in workflow_results:
        print(f"{workflow_result.workflow:<28} {'error: ' + str(workflow_result.error) if workflow_result.error else 'ok'}")
    if any(workflow_result.error for workflow_result in workflow_results):
        raise RuntimeError("GraphRAG indexing failed; see the workflow errors above and the logs folder.")
    save_indexing_report(
        run_inputs.run_directory,
        INDEX_NAME,
        indexing_time_seconds=time.perf_counter() - start_time,
        document_count=len(run_inputs.documents),
        corpus_token_count=int(run_inputs.documents["token_count"].sum()),
    )

indexing_cost_usd = ledger.total_cost_usd(INDEX_NAME, Phase.INDEXING)
print(f"Indexing cost: ${indexing_cost_usd:.4f}")

In [ ]:
from src.usage_tracking import project_full_run_cost_usd

if config.RUN_MODE is config.RunMode.SUBSET:
    projected_cost_usd = project_full_run_cost_usd(
        subset_cost_usd=indexing_cost_usd,
        subset_token_count=int(run_inputs.documents["token_count"].sum()),
        full_token_count=load_full_corpus_statistics(config.DOMAIN)["token_count"],
    )
    print(f"Projected GraphRAG indexing cost on the full corpus: ${projected_cost_usd:.2f}")

## 4.3 Look at the index

The index is a set of tables. Entities and relations form the graph; communities group entities; community reports summarise each group.

In [ ]:
from src.graphrag_utils import load_graphrag_index_tables

index_tables = await load_graphrag_index_tables(graphrag_config)
{table_name: len(table) for table_name, table in index_tables.items()}

In [ ]:
index_tables["entities"].sort_values("degree", ascending=False)[["title", "type", "degree", "description"]].head(10)

Communities are nested: level 0 holds a few large groups, deeper levels split them further. Search uses the reports up to `GRAPHRAG_COMMUNITY_LEVEL`.

In [ ]:
community_reports = index_tables["community_reports"]
print(community_reports.groupby("level").size().rename("reports per level").to_string())

largest_report = community_reports.sort_values("size", ascending=False).iloc[0]
print(f"\n{largest_report['title']} (level {largest_report['level']}, {largest_report['size']} entities)")
print(largest_report["summary"])

## 4.4 Local search

Local search needs the full set of tables. It returns the answer and the context it was built from.

In [ ]:
from graphrag.api import global_search, local_search

from src.question_runner import answer_all_questions


async def answer_with_local_search(question: str, question_type: str) -> str:
    """Answer one question with GraphRAG local search."""
    response, _ = await local_search(
        config=graphrag_config,
        entities=index_tables["entities"],
        communities=index_tables["communities"],
        community_reports=index_tables["community_reports"],
        text_units=index_tables["text_units"],
        relationships=index_tables["relationships"],
        covariates=None,
        community_level=config.GRAPHRAG_COMMUNITY_LEVEL,
        response_type=config.RESPONSE_TYPE,
        query=question,
    )
    return response


local_search_predictions = await answer_all_questions(
    answer_function=answer_with_local_search,
    questions=run_inputs.questions,
    system_name="graphrag_local",
    run_directory=run_inputs.run_directory,
    max_concurrent_questions=config.MAX_CONCURRENT_QUESTIONS,
)

## 4.5 Global search

Global search only needs entities, communities and reports: it never reads the raw chunks. Dynamic community selection is off, the default of the GraphRAG CLI, so every report up to the chosen level is read.

In [ ]:
async def answer_with_global_search(question: str, question_type: str) -> str:
    """Answer one question with GraphRAG global search."""
    response, _ = await global_search(
        config=graphrag_config,
        entities=index_tables["entities"],
        communities=index_tables["communities"],
        community_reports=index_tables["community_reports"],
        community_level=config.GRAPHRAG_COMMUNITY_LEVEL,
        dynamic_community_selection=False,
        response_type=config.RESPONSE_TYPE,
        query=question,
    )
    return response


global_search_predictions = await answer_all_questions(
    answer_function=answer_with_global_search,
    questions=run_inputs.questions,
    system_name="graphrag_global",
    run_directory=run_inputs.run_directory,
    max_concurrent_questions=config.MAX_CONCURRENT_QUESTIONS,
)

## 4.6 One question, two search modes

In [ ]:
example_question_id = run_inputs.questions.loc[run_inputs.questions["question_type"] == "summary", "question_id"].iloc[0]
example_question = run_inputs.questions.set_index("question_id").loc[example_question_id]
print("Q:", example_question["question"], "\n")
for system_name, predictions in [("graphrag_local", local_search_predictions), ("graphrag_global", global_search_predictions)]:
    answer_text = predictions.set_index("question_id").loc[example_question_id, "pred_answer"]
    print(f"--- {system_name}\n{answer_text[:700]}\n")

## 4.7 What it cost

Both search modes share one index. Their query costs differ: global search makes one map call per batch of reports for every question.

In [ ]:
ledger_summary = ledger.summarize_by_system_and_phase()
ledger_summary[ledger_summary["system_name"].isin([INDEX_NAME, "graphrag_local", "graphrag_global"])]

## Next

All four systems have answered. Notebook 05 grades the answers and puts quality and cost side by side.